# CONFIG

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


%load_ext autoreload
%autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/BIOM/notebooks
/home/turbotowerlnx/Documents/Master/BIOM/app


In [ ]:
import numpy as np
import os

from maikol_utils.print_utils import print_separator, print_warn, print_color

from src.config import Configuration    


CONFIG = Configuration(
    max_stages=15,
    objective_fpr=0.05
)

# seed all
np.random.seed(CONFIG.seed)

# GET ALL POSSIBLE HAAR

- We no dot create, for a single HAAR, both versions white-black / black-white, only one
- Since one is just the other $\times -1$, the weak classifiers will handle that by multiplying by $1$ or $-1$ as needed 

In [ ]:
from src.data import generate_all_features

all_features = generate_all_features(
    win_w = CONFIG.crop_size, 
    win_h = CONFIG.crop_size
)
len(all_features)

86400

# LOAD DATA

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from maikol_utils.file_utils import load_json, list_dir_files

# =============== FACES DATASET =============== 
all_faces, n = list_dir_files(CONFIG.faces_path, recursive=True)
print(f" - Found {n} files in {CONFIG.faces_path}")


# =============== NO-FACES DATASET =============== 
to_keep_labels = load_json(CONFIG.dataset_classes_path)

# Download dataset without faces
bg_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=to_keep_labels,
    max_samples=20000,
    dataset_dir=CONFIG.no_faces_path
)
bg_dataset = bg_dataset.filter_labels("ground_truth", F("label").is_in(to_keep_labels))


/home/turbotowerlnx/Documents/Master/BIOM/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 - Found 136965 files in ../data/ViolaJones/face_images
Loading output from ../data/dataset_classes.json...
Necessary images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'open-images-v7-train-5000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


### PRECOMPUTE ALL FEATURES FOR FACES DATASET

In [6]:
all_faces = np.random.choice(all_faces, size=5000, replace=False)
print(f"Using {len(all_faces)} files in {CONFIG.faces_path}")

Using 5000 files in ../data/ViolaJones/face_images


In [ ]:
from src.data import compute_features_dataset


if os.path.exists(CONFIG.faces_np_path):
    print(f" - Loading precomputed face features from {CONFIG.faces_np_path}...")
    X_train_faces = np.load(CONFIG.faces_np_path)
    print(f" - Loaded face features for {X_train_faces.shape[0]} images.")
    print(f" - X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")
else:
    X_train_faces = compute_features_dataset(all_faces, all_features)
    np.save(CONFIG.faces_np_path, X_train_faces)
    print(f" - Computed face features for {X_train_faces.shape[0]} images.")
    print(f" - X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")


Extracting face features (18 workers): 100%|██████████| 5000/5000 [00:03<00:00, 1346.80it/s]


 - Computed face features for 5000 images.
 - X_train_faces dtype=float32, shape=(5000, 86400)


# VIOLA JONES STAGES

### Train AdaBoost
- Train with stumps, using at most x features per stage
- Then check at which point of the trained, we mee the True positive and False positive rate
- Remove unused stages 
- Return remaining tree and threshold for that stage

In [ ]:
from src.model import train_stage_early_stopping

### Generate hard mining samples
- Load random images without faces and get crops that PASS all the current stages


In [ ]:
from src.data import balance_non_face_samples

### Generate all cascade stages
2 stopping criteria 
- reaching FPR of $10 e -6$. This is computed not as in the paper (they estimated it from each stage $fpr_i$), but rather from the number of windows that had to be scanned to create the new 'X_train_bg'.
- Hand number (say 50)


In [ ]:
from src.model import generate_all_stages

stages, fpr_macro = generate_all_stages(
    X_train_faces=X_train_faces,
    bg_samples=[sample.filepath for sample in bg_dataset],
    max_stages=CONFIG.max_stages,
    target_fpr=CONFIG.objective_fpr
)

________________________________________________________________________________________________________________________________
                                                      Training stage 1/15                                                       

________________________________________________________________
                Generating hard negative samples                

 - Generating 5000 hard negative samples (8 workers)...


Hard negatives: 100%|██████████| 5000/5000 [00:42<00:00, 117.28crops/s, crops=5000, imgs=1]


 - Collected 5000 crops from 1 images (target was 5000)
________________________________________________________________
                            Training                            

 - Fitting AdaBoost with early stopping...
 - Refining threshold and selecting features...
   - Features: 1 | FPR: 1.000
   - Features: 2 | FPR: 1.000
   - Features: 3 | FPR: 1.000
   - Features: 4 | FPR: 1.000
   - Features: 5 | FPR: 1.000
   - Features: 6 | FPR: 0.969
   - Features: 7 | FPR: 0.981
   - Features: 8 | FPR: 0.981
   - Features: 9 | FPR: 0.976
   - Features: 10 | FPR: 0.976
   - Features: 11 | FPR: 0.971
   - Features: 12 | FPR: 0.971
   - Features: 13 | FPR: 0.945
   - Features: 14 | FPR: 0.945
   - Features: 15 | FPR: 0.943
   - Features: 16 | FPR: 0.943
   - Features: 17 | FPR: 0.931
   - Features: 18 | FPR: 0.931
   - Features: 19 | FPR: 0.914
   - Features: 20 | FPR: 0.914
   - Features: 21 | FPR: 0.909
   - Features: 22 | FPR: 0.909
   - Features: 23 | FPR: 0.909
   - Features: 24 

Hard negatives:   0%|          | 1/2498 [16:00<666:11:48, 960.48s/crops, crops=1, imgs=5000]


⚠️Only found 1 crops (requested 2498) after processing 5000 images.⚠️
________________________________________________________________
                            Training                            

 - Fitting AdaBoost with early stopping...
 - Refining threshold and selecting features...
   - Features: 1 | FPR: 1.000
   - Features: 2 | FPR: 1.000
   - Features: 3 | FPR: 1.000
   - Features: 4 | FPR: 1.000
   - Features: 5 | FPR: 1.000
   - Features: 6 | FPR: 0.984
   - Features: 7 | FPR: 0.989
   - Features: 8 | FPR: 0.989
   - Features: 9 | FPR: 0.986
   - Features: 10 | FPR: 0.986
   - Features: 11 | FPR: 0.976
   - Features: 12 | FPR: 0.979
   - Features: 13 | FPR: 0.979
   - Features: 14 | FPR: 0.977
   - Features: 15 | FPR: 0.977
   - Features: 16 | FPR: 0.977
   - Features: 17 | FPR: 0.977
   - Features: 18 | FPR: 0.964
   - Features: 19 | FPR: 0.964
   - Features: 20 | FPR: 0.962
   - Features: 21 | FPR: 0.962
   - Features: 22 | FPR: 0.977
   - Features: 23 | FPR: 0.977
   -

Hard negatives:   0%|          | 0/3409 [15:54<?, ?crops/s, crops=0, imgs=5000]

⚠️Only found 0 crops (requested 3409) after processing 5000 images.⚠️


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 86400 and the array at index 1 has size 0

# FINAL STAGES ClASSIFIER

### Save the trained cascade to XML

In [ ]:
from src.model import CascadeSerializer, build_haar_cascade_from_stages

haar_cascade = build_haar_cascade_from_stages(
    stages_output=stages,
    all_features=all_features,
    width=CONFIG.crop_size,
    height=CONFIG.crop_size,
    cascade_type="trained_adaboost_stages",
    feature_type="HAAR",
)

print_separator(f"FINAL CASCADE", sep_type="LONG")
print(f" - Stages: {len(haar_cascade.stages)}")
print(f" - Features used: {len(haar_cascade.features)}")
print(f" - Window size: {haar_cascade.width}x{haar_cascade.height}")
print(haar_cascade)


cascade_path = os.path.join(CONFIG.computed_haar_cascades, f"stages_vj-{fpr_macro:.4f}.xml")
CascadeSerializer.save(haar_cascade, cascade_path)

print(f" - File: {cascade_path}")

In [ ]:
loaded_cascade = CascadeSerializer.load(cascade_path)